# Garbage Classification Demo Notebook

This notebook demonstrates loading the trained garbage classifier, evaluating the model, and running a sample prediction.

In [ ]:
import json
from pathlib import Path

import torch
from PIL import Image
from IPython.display import display

from src.data import get_transforms
from src.models import build_model

: 

## Load the model and checkpoint

In [ ]:
checkpoint_path = Path('outputs/checkpoints/best.pt')
assert checkpoint_path.exists(), 'Checkpoint not found: outputs/checkpoints/best.pt'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ckpt = torch.load(checkpoint_path, map_location=device)
model = build_model(ckpt['model_name'], len(ckpt['classes']), pretrained=False).to(device)
model.load_state_dict(ckpt['model_state'])
model.eval()
classes = ckpt['classes']

print('Loaded model:', ckpt['model_name'])
print('Classes:', classes)

## Run inference on a sample test image

In [ ]:
sample_path = Path('data/raw/Garbage classification/Garbage classification/glass/glass181.jpg')
assert sample_path.exists(), f'Sample image not found: {sample_path}'
image = Image.open(sample_path).convert('RGB')
display(image)
transform = get_transforms(ckpt.get('image_size', 224), train=False)
tensor = transform(image).unsqueeze(0).to(device)
with torch.no_grad():
    probs = model(tensor).softmax(dim=1)[0].cpu()
topk = torch.topk(probs, k=min(3, len(classes)))

print('Top predictions:')
for idx, prob in zip(topk.indices.tolist(), topk.values.tolist()):
    print(f'{classes[idx]}: {prob:.4f}')